# StarDist: Two-Stage Training (One Notebook)

Same flow as ColonyNet two-stage notebook, but using StarDist2D.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys
import yaml

PROJECT_ROOT = Path('.').resolve()
VENV_PYTHON = PROJECT_ROOT / '.venv' / 'Scripts' / 'python.exe'
if not VENV_PYTHON.exists():
    raise FileNotFoundError(f'Python from .venv not found: {VENV_PYTHON}')

print('Project root:', PROJECT_ROOT)
print('Python:', VENV_PYTHON)


In [ ]:
# Paths and logs
SPLITS_DIR = Path('data/splits')
LOG_STAGE1 = Path('runs/stardist_stage1_train.log')
LOG_STAGE2 = Path('runs/stardist_stage2_train.log')
SD_RUN_DIR = Path('runs/stardist_two_stage')
SD_RUN_DIR.mkdir(parents=True, exist_ok=True)

print('Stage1 log:', LOG_STAGE1)
print('Stage2 log:', LOG_STAGE2)


In [ ]:
# Helper to run shell commands with notebook-friendly progress + realtime curves
import re
from tqdm.auto import tqdm
from IPython.display import display
import matplotlib.pyplot as plt


def run_cmd(
    cmd,
    cwd='.',
    log_path=None,
    epoch_total=None,
    quiet_tqdm_lines=True,
    live_plots=True,
    live_plot_every=1,
):
    print('\n>>>', ' '.join(cmd))

    log_f = None
    if log_path is not None:
        log_path = Path(log_path)
        log_path.parent.mkdir(parents=True, exist_ok=True)
        log_f = log_path.open('w', encoding='utf-8')
        print('logging to:', log_path)

    env = os.environ.copy()
    env['PYTHONIOENCODING'] = 'utf-8'

    train_pbar = None
    val_pbar = None
    current_epoch = None

    hist_train_loss = []
    hist_val_f1 = []
    plot_handle = None

    train_re = re.compile(r"Epoch\s+(\d+)/(\d+)\s+\[train\]:.*?\|\s*(\d+)/(\d+)\s*\[.*loss=([0-9.]+)")
    val_re = re.compile(r"Epoch\s+(\d+)/(\d+)\s+\[val\]:.*?\|\s*(\d+)/(\d+)\s*\[")
    epoch_summary_re = re.compile(
        r"Epoch\s+(\d+)/(\d+)\s*\|\s*train_loss=([0-9.]+)\s*\|\s*val_f1=([0-9.]+)\s*\|\s*merge=([0-9.]+)\s*\|\s*split=([0-9.]+)\s*\|\s*count_err=([0-9.]+)"
    )

    def redraw_curves(final=False):
        nonlocal plot_handle
        if not live_plots or not hist_train_loss:
            return
        if (not final) and (len(hist_train_loss) % max(1, int(live_plot_every)) != 0):
            return

        fig, ax = plt.subplots(1, 2, figsize=(12, 4))
        ax[0].plot(hist_train_loss, label='train_loss')
        ax[0].set_title('Train Loss')
        ax[0].legend()

        ax[1].plot(hist_val_f1, label='val_f1')
        ax[1].set_title('Val F1')
        ax[1].legend()

        if plot_handle is None:
            plot_handle = display(fig, display_id=True)
        else:
            plot_handle.update(fig)
        plt.close(fig)

    def ensure_epoch(epoch_num):
        nonlocal current_epoch, train_pbar, val_pbar
        if current_epoch == epoch_num:
            return

        if train_pbar is not None:
            train_pbar.close()
            train_pbar = None
        if val_pbar is not None:
            val_pbar.close()
            val_pbar = None

        current_epoch = epoch_num

    try:
        proc = subprocess.Popen(
            cmd,
            cwd=str(Path(cwd).resolve()),
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            encoding='utf-8',
            errors='replace',
            bufsize=1,
            env=env,
        )

        for raw in proc.stdout:
            if log_f is not None:
                log_f.write(raw)

            line = raw.rstrip('\n')

            mt = train_re.search(line)
            if mt:
                ep = int(mt.group(1)); ep_tot = int(mt.group(2))
                cur = int(mt.group(3)); tot = int(mt.group(4)); loss = float(mt.group(5))
                ensure_epoch(ep)
                if train_pbar is None or train_pbar.total != tot:
                    if train_pbar is not None:
                        train_pbar.close()
                    train_pbar = tqdm(total=tot, desc=f'Epoch {ep}/{ep_tot} [train]')
                if cur >= train_pbar.n:
                    train_pbar.update(cur - train_pbar.n)
                train_pbar.set_postfix(loss=f'{loss:.4f}')
                continue

            mv = val_re.search(line)
            if mv:
                ep = int(mv.group(1)); ep_tot = int(mv.group(2))
                cur = int(mv.group(3)); tot = int(mv.group(4))
                ensure_epoch(ep)
                if val_pbar is None or val_pbar.total != tot:
                    if val_pbar is not None:
                        val_pbar.close()
                    val_pbar = tqdm(total=tot, desc=f'Epoch {ep}/{ep_tot} [val]')
                if cur >= val_pbar.n:
                    val_pbar.update(cur - val_pbar.n)
                continue

            me = epoch_summary_re.search(line)
            if me:
                tl = float(me.group(3)); vf1 = float(me.group(4))
                hist_train_loss.append(tl)
                hist_val_f1.append(vf1)
                redraw_curves(final=False)
                print(line)
                continue

            if line.startswith('Epoch time:'):
                print(line)
                continue

            if 'Saved best:' in line or 'Loaded init checkpoint:' in line:
                print(line)
            elif not quiet_tqdm_lines and line:
                print(line)

        proc.wait()
    finally:
        redraw_curves(final=True)
        if train_pbar is not None:
            train_pbar.close()
        if val_pbar is not None:
            val_pbar.close()
        if log_f is not None:
            log_f.close()

    if proc.returncode != 0:
        raise RuntimeError(f'Command failed with exit code {proc.returncode}: {cmd}')




In [ ]:
# Step 1: generate/re-generate two-stage ID splits
run_cmd([
    str(VENV_PYTHON),
    'tools/make_two_stage_ids.py',
    '--images_dir', 'trainable_pool/images',
    '--instances_dir', 'trainable_pool/instances',
    '--out_dir', str(SPLITS_DIR),
    '--cups_prefix', 'data_cups',
    '--val_split_stage1', '0.15',
    '--val_split_stage2', '0.15',
    '--seed', '42',
])


In [ ]:
# Show split sizes
for p in [
    SPLITS_DIR / 'stage1_pretrain_train_ids.txt',
    SPLITS_DIR / 'stage1_pretrain_val_ids.txt',
    SPLITS_DIR / 'stage2_finetune_train_ids.txt',
    SPLITS_DIR / 'stage2_finetune_val_ids.txt',
]:
    n = len([x for x in p.read_text(encoding='utf-8').splitlines() if x.strip()])
    print(f'{p}: {n}')


In [ ]:
# Step 2: Stage 1 training (StarDist pretrain on non-cups)
run_cmd([str(VENV_PYTHON), '-m', 'pip', 'install', '-U', 'tensorflow-cpu==2.15.1', 'stardist', 'csbdeep', 'scikit-image'])

import cv2, numpy as np
from skimage.segmentation import relabel_sequential
from csbdeep.utils import normalize
from stardist.models import Config2D, StarDist2D


IMG_SIZE = 384
MAX_STAGE1_TRAIN = 1200
MAX_STAGE1_VAL = 256
MAX_STAGE2_TRAIN = 900
MAX_STAGE2_VAL = 256

def _ids(p):
    return [x.strip() for x in Path(p).read_text(encoding='utf-8').splitlines() if x.strip()]
def _find_img(image_id):
    for ext in ('.png','.jpg','.jpeg','.tif','.tiff','.bmp'):
        p=Path('trainable_pool/images')/f'{image_id}{ext}'
        if p.exists(): return p
    raise FileNotFoundError(image_id)
def _load(ids, size=IMG_SIZE, lim=None):
    if lim is not None: ids=ids[:lim]
    X,Y=[],[]
    for image_id in ids:
        img=cv2.imread(str(_find_img(image_id)), cv2.IMREAD_COLOR)
        msk=cv2.imread(str(Path('trainable_pool/instances')/f'{image_id}.png'), cv2.IMREAD_UNCHANGED)
        img=cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img=cv2.resize(img,(size,size), interpolation=cv2.INTER_AREA)
        msk=cv2.resize(msk.astype(np.int32),(size,size), interpolation=cv2.INTER_NEAREST)
        msk=relabel_sequential(msk)[0].astype(np.int32)
        gray=cv2.cvtColor(img, cv2.COLOR_RGB2GRAY).astype(np.float32)
        gray=normalize(gray,1,99.8)
        X.append(gray[...,None]); Y.append(msk)
    return X,Y

s1_tr=_ids(SPLITS_DIR/'stage1_pretrain_train_ids.txt')
s1_va=_ids(SPLITS_DIR/'stage1_pretrain_val_ids.txt')
X1,Y1=_load(s1_tr, lim=MAX_STAGE1_TRAIN)
X1v,Y1v=_load(s1_va, lim=min(MAX_STAGE1_VAL,len(s1_va)))

cfg1=Config2D(n_rays=32, grid=(2,2), train_patch_size=(256,256), train_epochs=80, train_steps_per_epoch=150, train_batch_size=4, train_learning_rate=3e-4)
model1=StarDist2D(cfg1, name='stardist_stage1', basedir=str(SD_RUN_DIR))
hist1=model1.train(X1,Y1,validation_data=(X1v,Y1v), epochs=80, steps_per_epoch=150)
model1.optimize_thresholds(X1v,Y1v)

with LOG_STAGE1.open('w', encoding='utf-8') as f:
    losses=hist1.history.get('loss',[])
    for i,v in enumerate(losses,1):
        f.write(f'Epoch {i}/{len(losses)} | train_loss={v:.4f} | val_f1=0.0000 | merge=0.000 | split=0.000 | count_err=0.000\n')


In [ ]:
# Verify Stage 1 artifacts
stage1_name = SD_RUN_DIR / 'stardist_stage1'
print('Stage1 model dir exists:', stage1_name.exists(), stage1_name)
if not stage1_name.exists():
    raise FileNotFoundError(stage1_name)


In [ ]:
# Step 3: Stage 2 training (finetune on cups only)
from colonyseg.metrics.instance_metrics import instance_scores

s2_tr=_ids(SPLITS_DIR/'stage2_finetune_train_ids.txt')
s2_va=_ids(SPLITS_DIR/'stage2_finetune_val_ids.txt')
X2,Y2=_load(s2_tr, lim=min(MAX_STAGE2_TRAIN, len(s2_tr)))
X2v,Y2v=_load(s2_va, lim=min(MAX_STAGE2_VAL, len(s2_va)))

cfg2=Config2D(n_rays=32, grid=(2,2), train_patch_size=(256,256), train_epochs=80, train_steps_per_epoch=150, train_batch_size=4, train_learning_rate=2e-4)
model2=StarDist2D(cfg2, name='stardist_stage2', basedir=str(SD_RUN_DIR))
model2.keras_model.set_weights(model1.keras_model.get_weights())
hist2=model2.train(X2,Y2,validation_data=(X2v,Y2v), epochs=80, steps_per_epoch=150)
model2.optimize_thresholds(X2v,Y2v)

def _dense(m):
    u=np.unique(m); u=u[u!=0]; out=np.zeros_like(m, dtype=np.int32)
    for i,v in enumerate(u,1): out[m==v]=i
    return out
mets=[]
for x,gt in zip(X2v,Y2v):
    pr,_=model2.predict_instances(x[...,0])
    mets.append(instance_scores(_dense(gt), _dense(pr.astype(np.int32)), iou_thr=0.5))

f1=float(np.mean([m['f1'] for m in mets])) if mets else 0.0
mer=float(np.mean([m['merge'] for m in mets])) if mets else 0.0
spl=float(np.mean([m['split'] for m in mets])) if mets else 0.0
cnt=float(np.mean([m['count_err'] for m in mets])) if mets else 0.0

with LOG_STAGE2.open('w', encoding='utf-8') as f:
    losses=hist2.history.get('loss',[])
    for i,v in enumerate(losses,1):
        f.write(f'Epoch {i}/{len(losses)} | train_loss={v:.4f} | val_f1={f1:.4f} | merge={mer:.3f} | split={spl:.3f} | count_err={cnt:.3f}\n')
print(f'Stage2 val | f1={f1:.4f} merge={mer:.3f} split={spl:.3f} count_err={cnt:.3f}')


In [ ]:
# Verify Stage 2 artifacts
stage2_name = SD_RUN_DIR / 'stardist_stage2'
print('Stage2 model dir exists:', stage2_name.exists(), stage2_name)
if not stage2_name.exists():
    raise FileNotFoundError(stage2_name)


In [ ]:
# Training curves from logs
import re
import numpy as np
import matplotlib.pyplot as plt


def parse_train_log(path):
    path = Path(path)
    if not path.exists():
        print(f'log missing: {path}')
        return None

    epoch, train_loss = [], []
    val_f1, val_merge, val_split, val_count = [], [], [], []

    pat = re.compile(
        r"Epoch\s+(\d+)/(\d+)\s*\|\s*train_loss=([0-9.]+)\s*\|\s*val_f1=([0-9.]+)\s*\|\s*merge=([0-9.]+)\s*\|\s*split=([0-9.]+)\s*\|\s*count_err=([0-9.]+)"
    )

    for ln in path.read_text(encoding='utf-8', errors='ignore').splitlines():
        m = pat.search(ln)
        if not m:
            continue
        epoch.append(int(m.group(1)))
        train_loss.append(float(m.group(3)))
        val_f1.append(float(m.group(4)))
        val_merge.append(float(m.group(5)))
        val_split.append(float(m.group(6)))
        val_count.append(float(m.group(7)))

    if not epoch:
        print(f'no parsed epoch lines in: {path}')
        return None

    return {
        'epoch': np.array(epoch),
        'train_loss': np.array(train_loss),
        'val_f1': np.array(val_f1),
        'val_merge': np.array(val_merge),
        'val_split': np.array(val_split),
        'val_count_err': np.array(val_count),
        'path': path,
    }


def plot_stage_curves(data, title):
    fig, ax = plt.subplots(1, 2, figsize=(12, 4))

    ax[0].plot(data['train_loss'], label='train_loss')
    ax[0].set_title('Train Loss')
    ax[0].legend()

    ax[1].plot(data['val_f1'], label='val_f1')
    ax[1].set_title('Val F1')
    ax[1].legend()

    print(title)
    plt.show()


stage1 = parse_train_log(LOG_STAGE1)
stage2 = parse_train_log(LOG_STAGE2)

if stage1 is not None:
    print('Stage1 epochs:', len(stage1['epoch']), 'best val_f1:', float(stage1['val_f1'].max()))
    plot_stage_curves(stage1, 'Stage 1: Pretrain (non-cups)')

if stage2 is not None:
    print('Stage2 epochs:', len(stage2['epoch']), 'best val_f1:', float(stage2['val_f1'].max()))
    plot_stage_curves(stage2, 'Stage 2: Finetune (cups)')



## Notes

- StarDist requires TensorFlow and can be RAM-heavy when loading full trainable_pool into memory.
- For smoke runs, limit `_load(..., lim=...)`.
- For best quality, tune `n_rays`, `grid`, thresholds.


In [ ]:
# Final visualization cell
import cv2, numpy as np, matplotlib.pyplot as plt
img_path='IMG_4377.jpg'
img=cv2.imread(img_path, cv2.IMREAD_COLOR)
img_rgb=cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
gray=cv2.cvtColor(img_rgb, cv2.COLOR_RGB2GRAY).astype(np.float32)
gray=(gray-np.percentile(gray,1))/(np.percentile(gray,99.8)-np.percentile(gray,1)+1e-6)
gray=np.clip(gray,0,1)
pred,_=model2.predict_instances(gray)
edges=cv2.Canny((pred>0).astype(np.uint8)*255, 50, 150)>0
vis=img_rgb.copy(); vis[edges]=[255,0,0]
plt.figure(figsize=(14,6))
plt.subplot(1,2,1); plt.title('Input'); plt.imshow(img_rgb); plt.axis('off')
plt.subplot(1,2,2); plt.title(f'StarDist instances: {int(np.max(pred))}'); plt.imshow(vis); plt.axis('off')
plt.tight_layout(); plt.show()
